# (Our Code) Stage 4 — VACE personal-video run [AI-CODE] [HUMAN-CHECK]

**Each person:** File → Save a copy in Drive, then Runtime → Change runtime type → free T4 GPU. Run this notebook from the top in a fresh runtime. Upload **your own `stage4_sean_inputs.zip` or `stage4_ruijie_inputs.zip`**, prepared after your manual annotations and the final frozen pipeline. Your account runs your own generation. No paid compute or Hugging Face token is needed.

This notebook runs the actual model once. It does not run benchmarks. Keep the first completed output even if the artwork looks poor. Technical failures must be preserved and logged. Never provide hand-drawn audit masks to the model.

The recorded original and corrected runs are complete. The supplied staff response permits one descriptive-prompt correction with both runs retained. `PROMPT_REVISION=False` reproduces the original prompt and `True` selects the recorded descriptive prompt. Use the same fixed input ZIP and numeric settings. The modes save separate result folders and download names. Do not regenerate merely to seek a better result.


In [ ]:
import csv, gc, hashlib, io, json, platform, shutil, subprocess, sys, time, uuid, zipfile
from datetime import datetime, timezone
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
from IPython.display import display, Video
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one person\'s input ZIP.')
zip_name, zip_bytes = next(iter(uploaded.items()))
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
    run = json.loads(archive.read('run.json'))
    assert run['schema'] == 'cs4243-vace-inputs-v1'
    clip_id = run['clip_id']
    assert clip_id in ('stage4_sean', 'stage4_ruijie')
    input_dir = Path('/content/stage4_vace/inputs') / clip_id
    input_dir.mkdir(parents=True, exist_ok=True)
    for name in archive.namelist():
        target = (input_dir / name).resolve()
        if not target.is_relative_to(input_dir.resolve()):
            raise ValueError('Invalid ZIP path.')
    archive.extractall(input_dir)
del uploaded, zip_bytes
conditions = run['conditions']
assert conditions['clip_id'] == clip_id
for name, expected_hash in run['assets_sha256'].items():
    assert hashlib.sha256((input_dir / name).read_bytes()).hexdigest() == expected_hash, name
settings = conditions['model']['settings']
assert settings == {'width':832, 'height':480, 'num_frames':81, 'num_inference_steps':20,
                    'guidance_scale':5.0, 'conditioning_scale':1.0,
                    'negative_prompt':'', 'max_sequence_length':512}
assert conditions['model']['seed'] == 0 and conditions['model']['output_fps'] == 16
assert run['model_id'] == 'Wan-AI/Wan2.1-VACE-1.3B-diffusers'
assert run['model_revision'] == 'ec4d2cb062b548996b179d493fdd05340de702a1'
PROMPT_REVISION = False
scene_prompts = {
    'stage4_sean': 'A handheld video of a rectangular framed poster on a light-coloured wall displaying the reference artwork: large dark navy EXPLORE lettering on a cream background, an orange sun, dark navy mountain peaks, teal trees and a winding cream path. The camera moves around the poster, and a purple patterned object passes in front of it.',
    'stage4_ruijie': 'A handheld video of a rectangular board displaying the reference artwork: large dark-blue ORBIT lettering on a cream background, surrounded by coral-red and teal planets and stars. The board stands on a dark bed beside plush toys. The camera moves around the board, and a black padded object passes in front of it.'}
conditions = json.loads(json.dumps(conditions))
if PROMPT_REVISION:
    conditions['prompt']['template'] = scene_prompts[clip_id]
fixed_prompt = conditions['prompt']['template']
assert fixed_prompt == (scene_prompts[clip_id] if PROMPT_REVISION else 'Replace the artwork on the board in this video with the attached artwork.')
source = run['source']
assert len(run['sample_source_indices']) == settings['num_frames']
assert run['sample_source_indices'] == [source['interval_start'] + round(k * source['fps'] / 16) for k in range(81)]
assert run['sample_source_indices'][-1] == source['interval_end']
assert len(list((input_dir / 'masks').glob('mask_*.png'))) == source['interval_frames']
assert run['mask_origin'] == 'quadrilaterals from the final flow_assisted pipeline, not audit masks'
result_dir = Path('/content/stage4_vace/results') / ('prompt_revision' if PROMPT_REVISION else 'original') / clip_id
result_dir.mkdir(parents=True, exist_ok=True)
for name, data in [('run.json', run), ('conditions.json', conditions)]:
    p = result_dir / name
    content = json.dumps(data, indent=2) + '\n'
    if p.exists() and p.read_text() != content:
        raise RuntimeError('Different inputs already exist for this clip in this runtime. Use a fresh runtime.')
    p.write_text(content)
print('YOUR CLIP:', clip_id, '| source interval:', source['interval_start'], 'to', source['interval_end'])
print('Fixed model settings:', settings)
print('Prompt:', fixed_prompt)
display(Image.open(input_dir / 'artwork.png'))

# (Our Code) Prepare aligned video and masks [AI-CODE] [HUMAN-CHECK]

The source video and board masks use identical source indices. Sample 81 frames at 16 fps, resize to 832×480, then fill the white edit region with RGB 128. The artwork remains a separate reference image. No artwork is pasted into the generated output.


In [ ]:
"""Prepare aligned inpainting inputs without retaining the old board texture.

Pass only board masks exported from the frozen pipeline, never audit masks.
This helper prepares model inputs and does not generate or composite outputs.
"""

import numpy as np
from PIL import Image


def prepare_sequence(video_frames, board_masks, size=(832, 480)):
    """Return PIL source/mask sequences and a preflight summary.

    Input frames and masks must already use identical source timestamps.
    Binary white mask pixels mark regions to regenerate. Gray filling occurs
    after resizing so interpolation cannot reintroduce old board pixels.
    """
    if not video_frames or len(video_frames) != len(board_masks):
        raise ValueError("Source and board-mask frame counts must match and be nonzero.")
    prepared, masks, painted = [], [], []
    for index, (frame, mask) in enumerate(zip(video_frames, board_masks)):
        image = frame.convert("RGB") if isinstance(frame, Image.Image) else Image.fromarray(frame).convert("RGB")
        raw = np.asarray(mask)
        if raw.ndim == 3:
            if not (np.array_equal(raw[..., 0], raw[..., 1])
                    and np.array_equal(raw[..., 0], raw[..., 2])):
                raise ValueError(f"Mask {index} must be monochrome.")
            raw = raw[..., 0]
        if raw.dtype == np.bool_:
            raw = raw.astype(np.uint8) * 255
        if raw.ndim != 2 or raw.shape != (image.height, image.width):
            raise ValueError(f"Source/mask dimensions differ at frame {index}.")
        if not np.isin(raw, [0, 255]).all():
            raise ValueError(f"Mask {index} must contain only 0 and 255.")
        mask_image = Image.fromarray(raw.astype(np.uint8)).resize(size, Image.Resampling.NEAREST)
        mask_array = np.asarray(mask_image)
        source = np.asarray(image.resize(size, Image.Resampling.LANCZOS)).copy()
        original_resized = source.copy()
        regenerate = mask_array == 255
        source[regenerate] = 128
        if not (source[regenerate] == 128).all():
            raise ValueError(f"Original content remains in the edit region at frame {index}.")
        if not np.array_equal(source[~regenerate], original_resized[~regenerate]):
            raise ValueError(f"Preprocessing changed pixels outside the edit mask at frame {index}.")
        prepared.append(Image.fromarray(source))
        masks.append(mask_image)
        painted.append(int(regenerate.sum()))
    summary = {
        "frames": len(prepared), "size": list(size), "gray_rgb": [128, 128, 128],
        "mask_values": [0, 255], "mask_white_means": "regenerate",
        "regenerate_pixels_per_frame": painted,
        "old_content_removed_inside_masks": True,
        "outside_mask_unchanged_by_gray_fill": True,
        "timestamp_alignment": "must be supplied by the caller using nominal source timestamps",
    }
    return prepared, masks, summary


cap = cv2.VideoCapture(str(input_dir / 'interval.mp4'))
assert int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) == source['interval_frames']
assert abs(cap.get(cv2.CAP_PROP_FPS) - source['fps']) < 0.001
assert (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))) == (source['width'], source['height'])
wanted = set(run['sample_source_indices'])
video_frames, board_masks, seen = [], [], []
for relative in range(source['interval_frames']):
    ok, bgr = cap.read()
    assert ok, relative
    original_index = source['interval_start'] + relative
    if original_index in wanted:
        video_frames.append(Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))
        board_masks.append(Image.open(input_dir / 'masks' / f'mask_{original_index:06d}.png').convert('L'))
        seen.append(original_index)
cap.release()
assert seen == run['sample_source_indices']
video_frames, masks, input_check = prepare_sequence(video_frames, board_masks, size=(832, 480))
del board_masks
artwork = Image.open(input_dir / 'artwork.png').convert('RGB')
(result_dir / 'input_check.json').write_text(json.dumps(input_check, indent=2) + '\n')
print('PASS: aligned source/masks, binary masks, old artwork greyed out. No generation yet.')
for i in (0, 40, 80):
    print('Source index:', seen[i])
    display(video_frames[i].resize((416, 240)))

# (Our Code) Install pinned libraries and check the GPU [AI-CODE] [HUMAN-CHECK]

Install before importing Transformers or bitsandbytes. This cell downloads dependencies only. If Colab asks for a Hugging Face secret, cancel that request: the checkpoint is public.


In [ ]:
setup_started = time.perf_counter()
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout)
packages = ['diffusers==0.36.0', 'transformers==4.56.2', 'accelerate==1.10.1',
            'sentencepiece==0.2.1', 'ftfy==6.3.1', 'imageio-ffmpeg==0.6.0', 'bitsandbytes==0.50.2']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
import torch, diffusers, transformers, accelerate, bitsandbytes
assert torch.cuda.is_available(), 'Select a free GPU runtime before continuing.'
setup = {'model':run['model_id'], 'revision':run['model_revision'],
         'gpu':torch.cuda.get_device_name(0),
         'gpu_memory_gib':torch.cuda.get_device_properties(0).total_memory / 2**30,
         'python':platform.python_version(), 'torch':torch.__version__,
         'diffusers':diffusers.__version__, 'transformers':transformers.__version__,
         'accelerate':accelerate.__version__, 'bitsandbytes':bitsandbytes.__version__}
print(json.dumps(setup, indent=2))

# (Our Code) Load the model using the tested free-T4 memory setup [AI-CODE] [HUMAN-CHECK]

Load the text encoder in NF4, the temporary video transformer in int8 and the VAE in float32. Encode the fixed prompt, release the text encoder and temporary transformer, then load the final native-fp16 video transformer. No inference or timing benchmark is run here.


In [ ]:
from transformers import UMT5EncoderModel, BitsAndBytesConfig as TextQuantizationConfig
from diffusers import AutoencoderKLWan, WanVACEPipeline, WanVACETransformer3DModel, BitsAndBytesConfig as VideoQuantizationConfig

model_id, revision = run['model_id'], run['model_revision']
text_encoder = UMT5EncoderModel.from_pretrained(
    model_id, subfolder='text_encoder', revision=revision, token=False,
    quantization_config=TextQuantizationConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float32, bnb_4bit_use_double_quant=True),
    torch_dtype=torch.float32, device_map={'':0}, low_cpu_mem_usage=True)
transformer = WanVACETransformer3DModel.from_pretrained(
    model_id, subfolder='transformer', revision=revision, token=False,
    quantization_config=VideoQuantizationConfig(load_in_8bit=True),
    torch_dtype=torch.float32, low_cpu_mem_usage=True)
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder='vae', revision=revision,
                                      token=False, torch_dtype=torch.float32).to('cuda')
vae.enable_tiling()
pipe = WanVACEPipeline.from_pretrained(model_id, revision=revision, token=False,
    text_encoder=text_encoder, transformer=transformer, vae=vae, torch_dtype=torch.float32)
with torch.inference_mode():
    prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
        prompt=fixed_prompt, negative_prompt='', do_classifier_free_guidance=True,
        max_sequence_length=512, device=torch.device('cuda'))
assert torch.isfinite(prompt_embeds).all() and torch.isfinite(negative_prompt_embeds).all()
pipe.register_modules(text_encoder=None, transformer=None)
del text_encoder, transformer
gc.collect()
torch.cuda.empty_cache()
transformer = WanVACETransformer3DModel.from_pretrained(
    model_id, subfolder='transformer', revision=revision, token=False,
    torch_dtype=torch.float16, low_cpu_mem_usage=True).to('cuda')
pipe.register_modules(transformer=transformer)
setup['precision'] = {'transformer':'native fp16', 'vae':'float32 with tiling',
    'text_encoder':'NF4, double quantization, float32 compute, released after fixed-prompt encoding'}
setup['scheduler_class'] = type(pipe.scheduler).__name__
setup['scheduler_config'] = dict(pipe.scheduler.config)
setup['model_setup_s'] = time.perf_counter() - setup_started
print('MODEL READY for', clip_id, '| no personal-video generation has run yet.')

# (Our Code) Run the single fixed personal-video attempt [AI-CODE] [HUMAN-CHECK]

This cell performs the actual edit and logs the invocation before calling the model. **Do not rerun to improve a completed result.** If a technical failure occurs, run the download cell to preserve the error log. A technical retry must keep every setting and input unchanged and retain the first invocation. Set `TECHNICAL_RETRY=True` only for that case. If the runtime disconnects, a fresh session cannot recover files that were not downloaded.


In [ ]:
TECHNICAL_RETRY = False
from diffusers.utils import export_to_video

if (result_dir / 'output.mp4').exists():
    raise RuntimeError('A completed output already exists. Keep it and download it; do not regenerate.')
prior = sorted((result_dir / 'invocations').glob('*/run_metadata.json')) if (result_dir / 'invocations').exists() else []
if prior and not TECHNICAL_RETRY:
    raise RuntimeError('A prior invocation exists. Preserve its records. Only an unchanged technical retry is permitted.')
invocation_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
attempt_dir = result_dir / 'invocations' / invocation_id
attempt_dir.mkdir(parents=True)
log_fields = ['date','model','visible_version','settings','seed','interval','output_fps','prompt','inputs',
              'input_resolution','output_resolution','processing_time_s','compute','output_path','notes']
log_path = result_dir / 'genai_log.csv'
log_rows = []
if log_path.exists():
    with log_path.open(newline='') as f:
        log_rows = list(csv.DictReader(f))
actual_settings = {**settings, 'precision':setup['precision'], 'scheduler_class':setup['scheduler_class'],
                   'scheduler_config':setup['scheduler_config'], 'sample_source_indices':seen}
metadata = {'clip_id':clip_id, 'invocation_id':invocation_id, 'status':'started',
            'started_utc':datetime.now(timezone.utc).isoformat(), 'setup':setup,
            'conditions':conditions, 'actual_settings':actual_settings,
            'source_sha256':source['sha256'], 'artwork_sha256':run['artwork_sha256'],
            'technical_retry':bool(prior)}
row = {'date':metadata['started_utc'], 'model':model_id, 'visible_version':revision,
       'settings':json.dumps(actual_settings, sort_keys=True), 'seed':0,
       'interval':json.dumps({'start':source['interval_start'],'end':source['interval_end'],'clip_id':clip_id}),
       'output_fps':16, 'prompt':fixed_prompt,
       'inputs':json.dumps({'clip_id':clip_id,'input_zip':zip_name,'source_sha256':source['sha256'],
                           'board_masks':'frozen flow_assisted quadrilaterals','artwork_sha256':run['artwork_sha256']}),
       'input_resolution':f"{source['width']}x{source['height']}", 'output_resolution':'832x480',
       'processing_time_s':'', 'compute':f"Google Colab free {setup['gpu']}",
       'output_path':f"outputs/genai/{clip_id}/runs/{'revised_prompt' if PROMPT_REVISION else 'original_prompt'}/output.mp4",
       'notes':json.dumps({'clip_id':clip_id,'invocation_id':invocation_id,'status':'started','technical_retry':bool(prior)})}
log_rows.append(row)
def save_invocation():
    (attempt_dir / 'run_metadata.json').write_text(json.dumps(metadata, indent=2, default=str) + '\n')
    with log_path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=log_fields)
        writer.writeheader()
        writer.writerows(log_rows)
save_invocation()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
generation_started = time.perf_counter()
try:
    with torch.inference_mode():
        generated = pipe(video=video_frames, mask=masks, reference_images=artwork,
            prompt_embeds=prompt_embeds, negative_prompt_embeds=negative_prompt_embeds,
            height=480, width=832, num_frames=81, num_inference_steps=20,
            guidance_scale=5.0, conditioning_scale=1.0, max_sequence_length=512,
            generator=torch.Generator(device='cuda').manual_seed(0), output_type='np').frames[0]
    if not np.isfinite(generated).all():
        np.save(attempt_dir / 'nonfinite_output.npy', generated)
        raise RuntimeError('Non-finite model output saved for the technical record.')
    assert len(generated) == 81
    export_to_video(generated, str(attempt_dir / 'output.mp4'), fps=16)
    shutil.copyfile(attempt_dir / 'output.mp4', result_dir / 'output.mp4')
    metadata['output_sha256'] = hashlib.sha256((result_dir / 'output.mp4').read_bytes()).hexdigest()
    metadata['status'] = 'completed'
except Exception as error:
    metadata['status'] = 'technical_failure'
    metadata['error'] = repr(error)
    raise
finally:
    metadata['elapsed_s'] = time.perf_counter() - generation_started
    metadata['finished_utc'] = datetime.now(timezone.utc).isoformat()
    metadata['peak_allocated_gib'] = torch.cuda.max_memory_allocated() / 2**30
    metadata['peak_reserved_gib'] = torch.cuda.max_memory_reserved() / 2**30
    row['processing_time_s'] = f"{metadata['elapsed_s']:.3f}"
    row['notes'] = json.dumps({'clip_id':clip_id,'invocation_id':invocation_id,'status':metadata['status'],
        'technical_retry':bool(prior),'error':metadata.get('error'),
        'peak_allocated_gib':metadata['peak_allocated_gib'],'model_setup_s':setup['model_setup_s']})
    save_invocation()
print('COMPLETED. Keep this first output even if the edit is unsuccessful.')
display(Video(str(result_dir / 'output.mp4'), embed=True))

# (Our Code) Download the unchanged output and logs [AI-CODE] [HUMAN-CHECK]

Run this cell after success, or manually after a technical error, before disconnecting. Send the downloaded results ZIP back to Sean for local import and evaluation. Do not select more favourable output timestamps or repair the video.


In [ ]:
archive_path = Path('/content') / f"{clip_id}{'_prompt_revision' if PROMPT_REVISION else ''}_results.zip"
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for p in sorted(result_dir.rglob('*')):
        if p.is_file():
            z.write(p, (Path(clip_id) / p.relative_to(result_dir)).as_posix())
print('Download and preserve:', archive_path.name)
files.download(str(archive_path))